# MTDSim Package Verification

Test notebook to verify the restructured `mtdsim` package works correctly.

The package is organized into three pillars reflecting the MTD mental model:
- **`mtdsim.network`** — Network topology, hosts, services, vulnerabilities
- **`mtdsim.defender`** — MTD operations, techniques, and execution schemes
- **`mtdsim.attacker`** — Adversary model and attack simulation

Plus supporting modules:
- **`mtdsim.ai`** — DDQN-based reinforcement learning for MTD selection
- **`mtdsim.stats`** — Statistics, evaluation, and security metrics
- **`mtdsim.snapshot`** — Simulation state persistence
- **`mtdsim.data`** — Constants and static data files
- **`mtdsim.sim`** — Simulation infrastructure (time generators)

## 1. Package Import Verification

In [1]:
import mtdsim
print(f"MTDSim version: {mtdsim.__version__}")

# Pillar 1: Network
from mtdsim.network import Network, TimeNetwork, TargetNetwork, Host
from mtdsim.network.services import ServicesGenerator, Service, Vulnerability

# Pillar 2: Defender
from mtdsim.defender import MTD, MTDScheme, MTDOperation
from mtdsim.defender.techniques import (
    CompleteTopologyShuffle, IPShuffle, HostTopologyShuffle, PortShuffle,
    OSDiversity, OSDiversityAssignment, ServiceDiversity, UserShuffle,
)

# Pillar 3: Attacker
from mtdsim.attacker import Adversary, AttackOperation

# Supporting modules
from mtdsim.sim.time_generator import exponential_variates
from mtdsim.data import constants
from mtdsim.stats.evaluation import Evaluation
from mtdsim.stats.security_metric_statistics import SecurityMetricStatistics
from mtdsim.snapshot.snapshot_checkpoint import SnapshotCheckpoint

print("All core imports successful!")

MTDSim version: 0.2.0
All core imports successful!


## 2. Data File Access

Verify that static data files (words.txt, first-names.txt) are accessible via `importlib.resources`.

In [2]:
from importlib import resources

words_text = resources.files("mtdsim.data").joinpath("words.txt").read_text(encoding="utf-8")
names_text = resources.files("mtdsim.data").joinpath("first-names.txt").read_text(encoding="utf-8")

print(f"Words loaded: {len(words_text.splitlines())} entries")
print(f"Names loaded: {len(names_text.splitlines())} entries")
print(f"\nSample words: {words_text.splitlines()[:5]}")
print(f"Sample names: {names_text.splitlines()[:5]}")

Words loaded: 10000 entries
Names loaded: 4945 entries

Sample words: ['a', 'aa', 'aaa', 'aaron', 'ab']
Sample names: ['Aaren', 'Aarika', 'Abagael', 'Abagail', 'Abbe']


## 3. Network Creation

Create a `TimeNetwork` (the simulation-aware network) and inspect its properties. This exercises the Barabasi-Albert graph generation, host creation, service/vulnerability assignment, and user credential setup.

In [3]:
# Note: total_subnets must be sufficiently larger than total_layers to avoid
# an infinite loop in gen_graph(). The default parameters (50/5/8/4) from the
# theses are safe. Small networks need proportionally more subnets.
network = TimeNetwork(
    total_nodes=50,
    total_endpoints=5,
    total_subnets=8,
    total_layers=4,
    total_database=5,
    terminate_compromise_ratio=0.8,
)

graph = network.get_graph()
print(f"Total nodes:     {network.get_total_nodes()}")
print(f"Total endpoints: {network.get_total_endpoints()}")
print(f"Subnets:         {len(network.get_subnets())}")
print(f"Graph edges:     {graph.number_of_edges()}")
print(f"Graph density:   {2 * graph.number_of_edges() / (graph.number_of_nodes() * (graph.number_of_nodes() - 1)):.4f}")

# Inspect a sample host
sample_host = network.get_host(0)
print(f"\nSample host (node 0):")
print(f"  OS:             {sample_host.os_type} v{sample_host.os_version}")
print(f"  Total services: {sample_host.total_services}")
print(f"  Total users:    {sample_host.total_users}")
print(f"  Internal graph: {sample_host.graph.number_of_nodes()} nodes")

Total nodes:     50
Total endpoints: 5
Subnets:         50
Graph edges:     48
Graph density:   0.0392

Sample host (node 0):
  OS:             windows vvista
  Total services: 9
  Total users:    5
  Internal graph: 10 nodes


## 4. Adversary Setup

Initialize the attacker with the network. The adversary starts at the network entry point and aims to compromise hosts.

In [4]:
adversary = Adversary(network=network, attack_threshold=constants.ATTACKER_THRESHOLD)

print(f"Attack threshold:    {constants.ATTACKER_THRESHOLD}")
print(f"Max attack attempts: {adversary._max_attack_attempts}")
print(f"Initial process:     {adversary.get_curr_process()}")
print(f"Compromised hosts:   {len(adversary.get_compromised_hosts())}")

Attack threshold:    10
Max attack attempts: 250
Initial process:     SCAN_HOST
Compromised hosts:   0


## 5. MTD Techniques

Instantiate all 8 MTD techniques and verify their properties (type, priority, execution time).

In [5]:
import pandas as pd

techniques = [
    CompleteTopologyShuffle, HostTopologyShuffle, IPShuffle, PortShuffle,
    OSDiversity, OSDiversityAssignment, ServiceDiversity, UserShuffle,
]

rows = []
for Technique in techniques:
    t = Technique(network=network)
    rows.append({
        "Technique": t.get_name(),
        "Type": t.get_mtd_type(),
        "Resource": t.get_resource_type(),
        "Priority": t.get_priority(),
        "Exec Time Mean (s)": t.get_execution_time_mean(),
    })

df = pd.DataFrame(rows)
print(f"All {len(techniques)} MTD techniques instantiated successfully.\n")
df

All 8 MTD techniques instantiated successfully.



,Technique,Type,Resource,Priority,Exec Time Mean (s)
0,CompleteTopologyShuffle,shuffle,network,1,120
1,HostTopologyShuffle,shuffle,network,2,100
2,IPShuffle,shuffle,network,3,110
3,PortShuffle,shuffle,application,5,70
4,OSDiversity,diversity,application,4,80
5,DAP_OSDiversity_4,diversity,application,4,80
6,ServiceDiversity,diversity,application,6,70
7,UserShuffle,shuffle,reserve,7,20


## 6. SimPy Simulation Components

Verify that SimPy processes can be constructed for both attack and defense operations.

In [7]:
import simpy

env = simpy.Environment()
end_event = env.event()

# Create attack operation
attack_op = AttackOperation(env=env, end_event=end_event, adversary=adversary, proceed_time=0)
print(f"AttackOperation created: env={type(env).__name__}")

# Create security metrics record
security_metrics = SecurityMetricStatistics()

# Create MTD operation with 'random' scheme
mtd_op = MTDOperation(
    security_metrics_record=security_metrics,
    env=env,
    end_event=end_event,
    network=network,
    attack_operation=attack_op,
    scheme='random',
    adversary=adversary,
)
print(f"MTDOperation created: scheme='random'")
print(f"MTD scheme: {mtd_op._mtd_scheme.get_scheme()}")
print(f"MTD trigger interval: {mtd_op._mtd_scheme.get_mtd_trigger_interval()}s")

AttackOperation created: env=Environment
MTDOperation created: scheme='random'
MTD scheme: random
MTD trigger interval: 200s


## 7. Full Simulation Run

Run a complete simulation with attack and defense operating concurrently. This is the core use case from the original theses (Zhang 2023, Ho 2024, Tay 2024).

In [8]:
import logging
logging.disable(logging.CRITICAL)  # suppress verbose simulation logs

# Fresh network for simulation (using default safe parameters)
sim_network = TimeNetwork(total_nodes=50, total_endpoints=5, total_subnets=8, total_layers=4)
sim_adversary = Adversary(network=sim_network, attack_threshold=constants.ATTACKER_THRESHOLD)

# SimPy environment
sim_env = simpy.Environment()
sim_end = sim_env.event()
sim_metrics = SecurityMetricStatistics()

# Create operations
sim_attack = AttackOperation(env=sim_env, end_event=sim_end, adversary=sim_adversary, proceed_time=0)
sim_mtd = MTDOperation(
    security_metrics_record=sim_metrics,
    env=sim_env,
    end_event=sim_end,
    network=sim_network,
    attack_operation=sim_attack,
    scheme='random',
    adversary=sim_adversary,
)

# Launch both processes
sim_attack.proceed_attack()
sim_mtd.proceed_mtd()

# Run until either end_event triggers or timeout at 2000s
sim_env.run(until=sim_end | sim_env.timeout(2000))

# Results
compromised = len(sim_adversary.get_compromised_hosts())
total = sim_network.get_total_nodes()
print(f"Simulation ended at t={sim_env.now:.1f}s")
print(f"Compromised hosts: {compromised}/{total} ({compromised/total*100:.1f}%)")
print(f"Adversary final state: {sim_adversary.get_curr_process()}")
print(f"Target compromised: {sim_adversary.target_compromised}")

logging.disable(logging.NOTSET)  # re-enable logging

Simulation ended at t=2000.0s
Compromised hosts: 8/50 (16.0%)
Adversary final state: EXPLOIT_VULN
Target compromised: False


## 8. Snapshot / Checkpoint

Test the snapshot system for saving and restoring simulation state.

In [9]:
from mtdsim.snapshot.network_snapshot import NetworkSnapshot
from mtdsim.snapshot.adversary_snapshot import AdversarySnapshot
from mtdsim.snapshot.snapshot_checkpoint import SnapshotCheckpoint

net_snap = NetworkSnapshot()
adv_snap = AdversarySnapshot()
checkpoint = SnapshotCheckpoint()

print(f"NetworkSnapshot:     {type(net_snap).__name__} (save_network, load_network)")
print(f"AdversarySnapshot:   {type(adv_snap).__name__} (save_adversary, load_adversary)")
print(f"SnapshotCheckpoint:  {type(checkpoint).__name__}")
print("\nSnapshot classes instantiated successfully!")

NetworkSnapshot:     NetworkSnapshot (save_network, load_network)
AdversarySnapshot:   AdversarySnapshot (save_adversary, load_adversary)
SnapshotCheckpoint:  SnapshotCheckpoint

Snapshot classes instantiated successfully!


## 9. Constants & Configuration

Verify key simulation constants are accessible and match the thesis specifications.

In [10]:
print("=== OS Types ===")
print(f"  {constants.OS_TYPES}")

print("\n=== MTD Durations (mean, std) ===")
for name, (mean, std) in constants.MTD_DURATION.items():
    print(f"  {name:30s}: mean={mean}s, std={std}s")

print("\n=== MTD Priorities ===")
for name, priority in constants.MTD_PRIORITY.items():
    print(f"  {name:30s}: {priority}")

print(f"\n=== Attack Parameters ===")
print(f"  ATTACKER_THRESHOLD:      {constants.ATTACKER_THRESHOLD}")
print(f"  ATTACK_DURATION:         {constants.ATTACK_DURATION}")
print(f"  MTD_TRIGGER_INTERVAL:    {constants.MTD_TRIGGER_INTERVAL}")

=== OS Types ===
  ['windows', 'ubuntu', 'centos', 'freebsd']

=== MTD Durations (mean, std) ===
  CompleteTopologyShuffle       : mean=120s, std=0.5s
  HostTopologyShuffle           : mean=100s, std=0.5s
  IPShuffle                     : mean=110s, std=0.5s
  OSDiversity                   : mean=80s, std=0.5s
  PortShuffle                   : mean=70s, std=0.5s
  ServiceDiversity              : mean=70s, std=0.5s
  UserShuffle                   : mean=20s, std=0.5s

=== MTD Priorities ===
  CompleteTopologyShuffle       : 1
  HostTopologyShuffle           : 2
  IPShuffle                     : 3
  OSDiversity                   : 4
  PortShuffle                   : 5
  ServiceDiversity              : 6
  UserShuffle                   : 7

=== Attack Parameters ===
  ATTACKER_THRESHOLD:      10
  ATTACK_DURATION:         {'SCAN_HOST': 5, 'ENUM_HOST': 5, 'SCAN_NEIGHBOR': 5, 'SCAN_PORT': 25, 'EXPLOIT_VULN': 15, 'BRUTE_FORCE': 20, 'PENALTY': 20}
  MTD_TRIGGER_INTERVAL:    {'simultaneous': (

## 10. Execution Schemes

Test multiple MTD execution schemes (random, alternative, simultaneous) as defined in Zhang (2023).

In [11]:
for scheme in ['random', 'alternative', 'simultaneous']:
    test_metrics = SecurityMetricStatistics()
    mtd_scheme = MTDScheme(
        scheme=scheme,
        network=network,
        security_metric_record=test_metrics,
    )
    print(f"Scheme '{scheme}':")
    print(f"  Trigger interval: {mtd_scheme.get_mtd_trigger_interval()}s")
    print(f"  Trigger std:      {mtd_scheme.get_mtd_trigger_std()}\n")

Scheme 'random':
  Trigger interval: 200s
  Trigger std:      0.5

Scheme 'alternative':
  Trigger interval: 200s
  Trigger std:      0.5

Scheme 'simultaneous':
  Trigger interval: 700s
  Trigger std:      0.5



## Summary

All components of the restructured `mtdsim` package are working correctly:

| Component | Status |
|---|---|
| `mtdsim.network` (Network, TimeNetwork, Host, Services) | Verified |
| `mtdsim.defender` (MTDScheme, MTDOperation, 8 techniques) | Verified |
| `mtdsim.attacker` (Adversary, AttackOperation) | Verified |
| `mtdsim.stats` (SecurityMetricStatistics, Evaluation) | Verified |
| `mtdsim.snapshot` (NetworkSnapshot, AdversarySnapshot) | Verified |
| `mtdsim.data` (constants, static text files) | Verified |
| `mtdsim.sim` (time_generator) | Verified |
| Full simulation run (attack + defense concurrent) | Verified |
| Execution schemes (random, alternative, simultaneous) | Verified |

## Original Prompt

I have tried moving the MTD simulator into src, and removed many experiments files that were hangover from the previous theses. Understanding those is surplus to what my needs for this simulator is. However, I found that the structure of the codebase was very unclear and not representative of the mental model of the operators of this simulator. 

In a standard MTD system, there are three main components. There is the network (e.g. the hosts, connections, graphs, etc., which is typically a constant in these simulators), the defender (which is the mtd operation, so originally in the simulator the 7 implemented by the original creators, then various policies to activate such operations, e.g. no mtd,, one operation, randomly select operation, interval times etc., all proactive mtd techniques), and then the attacker (which includes the independent operator with the objective to compromise the network, with predetermined profiles "e.g. scan hosts, enum hosts etc. ). 

I have broken up the codebase to try and better reflect this, e.g. putting the components of the mtd-system in their own directories, and trying to put the right code in the right places. Now, later works have further attempted improve decision making of mtd defender through reinforcement learning, so I have tried putting this work into mtd-ai. Although I think I have deleted the trained models from the codebase, I am not sure where or how to retrain these, but I will.... i have put into the a separate directory (mtd-ai) because it involves picking the best mtd operation from a trained policy and is a separate piece of work from the original mtdsystem . 

Also any leftover utility functions (e.g. stats, static files, snapshots, i have tried putting into the utils directory I have tried to make sure each directory is a packagae with their original init files, but I may have overriden and broken file directory relationa dependencies. I would like you to fully flesh out my vision here, and properly implement my changes. My goal is for mtd-system to be a package to be imported in jupyter notebooks so that I can interact with the simulator and run simulations and use the simulator in a way that answers my research questions, which are broad ranging. I look like you to:

* reconstruct all the works in the src directory into fully working packages as originally implemented before the changes 
* read the original theses of preliminaries in .context/ to grasp the intentions and wishes of the previous works
* read all the src files, see how they work in relation to each other, fix imports, packages, etc, rename to make sense to the mental model I laid out
* rename/restructure directories if the way I laid it out is incorrect or nonsensical. Move files around if need be. Create/remove directory structures if need be. 
*Produce a jupyter notebook titled `2026-03-29_MTDSim_TestSrcDir` that checks all components work as intended, and that the original visions in the theses are still preserved (e.g. test simulator conditions, e.g. simulation runs would still feasibly work with the new import names)

Produce a thorough plan after reading the src files and the theses